# Event study

Market-model abnormal returns around the three dates that matter: 16-Jan-2026 (the date Sun Pharma's own press release cites as "before news of takeover interest emerged" - referred to here as the reference date rather than presumed to be a discrete leak, pending the tests below), 26-Apr-2026 (announcement), and 23-Jul-2026 (shareholder approval). All single-ticker return series are built from each ticker's own trading calendar directly - not from the multi-ticker panel - for the reason noted in the market-risk notebook: shifting by row position on a unioned calendar can silently drop a real trading day's return whenever another market's date sits in between.

In [1]:

import sys, warnings
sys.path.insert(0, r"/Users/shaan/Desktop/FAM/sunpharma-organon-merger-arbitrage")
warnings.filterwarnings("ignore")

from src import config as cfg, data, eventstudy as es, checks
import pandas as pd
import numpy as np
pd.set_option("display.width", 160)

prices = data.load_prices()

def own_returns(ticker):
    c = prices[ticker]["Close"]
    return np.log(c / c.shift(1)).dropna()

ogn_ret = own_returns(cfg.TARGET)
sun_ret = own_returns(cfg.ACQUIRER)
gspc_ret = own_returns(cfg.US_INDEX)
nsei_ret = own_returns(cfg.IN_INDEX)


## Market model estimation

Estimation window ends the day before the leak date and runs back 250 trading days - the same window used for the valuation beta, and for the same reason: a window anchored to the 26-Apr announcement would sit on top of the 16-Jan-to-26-Apr run-up and contaminate alpha and beta with the event itself.

In [2]:

window_end = pd.Timestamp(cfg.ESTIMATION_WINDOW_END)
checks.check_beta_window_uncontaminated(cfg.ESTIMATION_WINDOW_END)

ogn_est_dates = ogn_ret.index[ogn_ret.index <= window_end][-cfg.ESTIMATION_WINDOW_LENGTH:]
sun_est_dates = sun_ret.index[sun_ret.index <= window_end][-cfg.ESTIMATION_WINDOW_LENGTH:]

ogn_model = es.estimate_market_model(ogn_ret, gspc_ret, ogn_est_dates)
sun_model = es.estimate_market_model(sun_ret, nsei_ret, sun_est_dates)

print("OGN market model (vs S&P 500)")
print(f"  alpha(daily)={ogn_model['alpha']:.5f}  beta={ogn_model['beta']:.3f}  "
      f"resid_std={ogn_model['resid_std']:.4f}  n={ogn_model['n_obs']}")
print("Sun Pharma market model (vs Nifty 50)")
print(f"  alpha(daily)={sun_model['alpha']:.5f}  beta={sun_model['beta']:.3f}  "
      f"resid_std={sun_model['resid_std']:.4f}  n={sun_model['n_obs']}")


OGN market model (vs S&P 500)
  alpha(daily)=-0.00309  beta=0.969  resid_std=0.0395  n=250
Sun Pharma market model (vs Nifty 50)
  alpha(daily)=-0.00046  beta=0.723  resid_std=0.0115  n=249


In [3]:

ogn_ar = es.abnormal_returns(ogn_ret, gspc_ret, ogn_model)
sun_ar = es.abnormal_returns(sun_ret, nsei_ret, sun_model)


## Cumulative abnormal returns

Three event dates, three windows each, computed separately for the target (Organon) and the acquirer (Sun Pharma). The M&A literature's base rate is that acquirer CAR is muted or negative around an announcement (Roll's hubris hypothesis; Andrade, Mitchell & Stafford, 2001) - that is a prior to test against, not an assumed result; the table below is read as computed, not fitted to that expectation.

In [4]:

events = {
    "leak (16-Jan-2026)": pd.Timestamp(cfg.DEAL["leak_date"]),
    "announcement (26-Apr-2026)": pd.Timestamp(cfg.DEAL["announcement_date"]),
    "approval (23-Jul-2026)": pd.Timestamp(cfg.DEAL["shareholder_approval_date"]),
}

def car_table(ar, all_dates, resid_std, label):
    rows = []
    for ev_label, ev_date in events.items():
        for pre, post in cfg.EVENT_WINDOWS:
            wdates = es.event_window_dates(all_dates, ev_date, pre, post)
            res = es.car_with_tstat(ar, wdates, resid_std)
            rows.append({"entity": label, "event": ev_label, "window": f"({pre},{post})",
                         "CAR": res["CAR"], "n_days": res["n_days"],
                         "t_stat": res["t_stat"], "p_value": res["p_value"]})
    return pd.DataFrame(rows)

ogn_car = car_table(ogn_ar, ogn_ret.index, ogn_model["resid_std"], "Organon (target)")
sun_car = car_table(sun_ar, sun_ret.index, sun_model["resid_std"], "Sun Pharma (acquirer)")

pd.concat([ogn_car, sun_car]).set_index(["entity","event","window"]).round(4)


CAR  n_days  t_stat  p_value
entity                event                      window                                   
Organon (target)      leak (16-Jan-2026)         (-1,1)    0.1217       3  1.7814   0.2168
                                                 (-5,5)    0.1427      11  1.0902   0.3012
                                                 (-10,10)  0.1807      21  0.9995   0.3295
                      announcement (26-Apr-2026) (-1,1)    0.4426       3  6.4765   0.0230
                                                 (-5,5)    0.3303      11  2.5241   0.0302
                                                 (-10,10)  0.3955      21  2.1871   0.0408
                      approval (23-Jul-2026)     (-1,1)    0.0226       3  0.3313   0.7719
                                                 (-5,5)    0.0558      11  0.4266   0.6787
                                                 (-10,10)  0.0385      20  0.2180   0.8297
Sun Pharma (acquirer) leak (16-Jan-2026)         (-1,1)   -0.0118       2 -0.7235   0.6013
                                                 (-5,5)   -0.0479      10 -1.3110   0.2223
                                                 (-10,10) -0.0166      20 -0.3220   0.7509
                      announcement (26-Apr-2026) (-1,1)    0.0459       3  2.2968   0.1485
                                                 (-5,5)    0.0962      10  2.6361   0.0271
                                                 (-10,10)  0.0947      20  1.8341   0.0823
                      approval (23-Jul-2026)     (-1,1)    0.0034       3  0.1697   0.8809
                                                 (-5,5)    0.0230      11  0.6011   0.5612
                                                 (-10,10)  0.0199      21  0.3757   0.7111

In [5]:

sun_ann_5 = sun_car[(sun_car["event"]=="announcement (26-Apr-2026)") & (sun_car["window"]=="(-5,5)")].iloc[0]
ogn_leak_5 = ogn_car[(ogn_car["event"]=="leak (16-Jan-2026)") & (ogn_car["window"]=="(-5,5)")].iloc[0]

print("Reading the table above against the hubris-hypothesis prior:")
print(f"  Sun Pharma CAR, announcement (-5,+5): {sun_ann_5['CAR']:+.2%}, t={sun_ann_5['t_stat']:.2f}, p={sun_ann_5['p_value']:.3f}")
print(f"  -> positive and statistically significant. The market moved Sun Pharma's own price UP")
print(f"     around the announcement, not down - the opposite of the muted/negative acquirer")
print(f"     reaction the hubris-hypothesis base rate would predict. Read together with the")
print(f"     valuation notebook's finding that the offer requires a plausible (not extreme)")
print(f"     revenue-growth reversal, this is a second, independent signal against a simple")
print(f"     'Sun Pharma overpaid' story.")
print()
print(f"  Organon CAR, leak window (-5,+5): {ogn_leak_5['CAR']:+.2%}, t={ogn_leak_5['t_stat']:.2f}, p={ogn_leak_5['p_value']:.3f}")
print(f"  -> NOT statistically significant at 5% (p={ogn_leak_5['p_value']:.2f}). The narrow window")
print(f"     around 16-Jan-2026 does not show a sharp, statistically distinguishable jump - see")
print(f"     the volume test and the drift decomposition below for what the data actually supports.")


Reading the table above against the hubris-hypothesis prior:
  Sun Pharma CAR, announcement (-5,+5): +9.62%, t=2.64, p=0.027
  -> positive and statistically significant. The market moved Sun Pharma's own price UP
     around the announcement, not down - the opposite of the muted/negative acquirer
     reaction the hubris-hypothesis base rate would predict. Read together with the
     valuation notebook's finding that the offer requires a plausible (not extreme)
     revenue-growth reversal, this is a second, independent signal against a simple
     'Sun Pharma overpaid' story.

  Organon CAR, leak window (-5,+5): +14.27%, t=1.09, p=0.301
  -> NOT statistically significant at 5% (p=0.30). The narrow window
     around 16-Jan-2026 does not show a sharp, statistically distinguishable jump - see
     the volume test and the drift decomposition below for what the data actually supports.


## Abnormal trading volume around the leak date

A volume spike ahead of the reference date, without a matching price-only explanation, would be the standard corroborating signal for a discrete information-leakage event. Tested here rather than assumed.

In [6]:

ogn_volume = prices[cfg.TARGET]["Volume"]
vol_result = es.abnormal_volume(ogn_volume, pd.Timestamp(cfg.DEAL["leak_date"]), baseline_window=60, event_pre=10, event_post=0)

print(f"baseline mean daily volume (60d, ending 10d pre-leak) : {vol_result['baseline_mean_volume']:,.0f}")
print(f"event-window mean daily volume (t-10 to t0)            : {vol_result['event_mean_volume']:,.0f}")
print(f"ratio                                                    : {vol_result['ratio']:.2f}x")
print(f"z-score                                                  : {vol_result['z_score']:.2f}")
print()
print("No abnormal volume around 16-Jan-2026 (ratio ~1.0x, z~0.05). Combined with the")
print("non-significant narrow-window CAR above, this data does not support a discrete,")
print("single-day leakage event on that date. What it does support - see below - is a")
print("large, gradual price drift accruing over the three months that followed it.")


baseline mean daily volume (60d, ending 10d pre-leak) : 5,952,090
event-window mean daily volume (t-10 to t0)            : 6,208,718
ratio                                                    : 1.04x
z-score                                                  : 0.05

No abnormal volume around 16-Jan-2026 (ratio ~1.0x, z~0.05). Combined with the
non-significant narrow-window CAR above, this data does not support a discrete,
single-day leakage event on that date. What it does support - see below - is a
large, gradual price drift accruing over the three months that followed it.


## Pre-announcement price drift

How much of the total move from the unaffected price to the post-announcement price happened before Sun Pharma's 26-Apr press release, versus on the announcement itself. Reported on both a simple-return and log-return basis, since they differ by several points and this notebook should be explicit about which is which rather than picking one silently. This drift is real and large regardless of the volume/CAR result above - what that result rules out is a sharp, statistically detectable jump concentrated on the reference date itself; the move instead accrued gradually across the window.

In [7]:

ogn_close = prices[cfg.TARGET]["Close"]
p_unaffected = ogn_close.loc[ogn_close.index <= pd.Timestamp(cfg.DEAL["unaffected_date"])].iloc[-1]
p_pre_ann = ogn_close.loc[ogn_close.index <= pd.Timestamp("2026-04-24")].iloc[-1]
p_post_ann = ogn_close.loc[ogn_close.index >= pd.Timestamp("2026-04-27")].iloc[0]

move_pre_simple = p_pre_ann - p_unaffected
move_ann_simple = p_post_ann - p_pre_ann
move_total_simple = p_post_ann - p_unaffected
pct_pre_simple = move_pre_simple / move_total_simple

move_pre_log = np.log(p_pre_ann / p_unaffected)
move_ann_log = np.log(p_post_ann / p_pre_ann)
move_total_log = np.log(p_post_ann / p_unaffected)
pct_pre_log = move_pre_log / move_total_log

print(f"unaffected (9-Apr)     : ${p_unaffected:.2f}")
print(f"pre-announcement (24-Apr): ${p_pre_ann:.2f}")
print(f"post-announcement (27-Apr): ${p_post_ann:.2f}")
print()
print(f"simple-return basis : {pct_pre_simple:.1%} of the total move happened before the announcement "
      f"(+{move_pre_simple/p_unaffected:.1%} pre-announcement vs. +{move_ann_simple/p_pre_ann:.1%} on the announcement day)")
print(f"log-return basis    : {pct_pre_log:.1%} of the total log-move happened before the announcement")


unaffected (9-Apr)     : $6.91
pre-announcement (24-Apr): $11.26
post-announcement (27-Apr): $13.16

simple-return basis : 69.6% of the total move happened before the announcement (+63.0% pre-announcement vs. +16.9% on the announcement day)
log-return basis    : 75.8% of the total log-move happened before the announcement


## Peer spillover

CAR for each US pharma peer around the 26-Apr announcement window, testing whether the deal moved comparable stocks (a read-through / re-rating effect) or was idiosyncratic to Organon and Sun Pharma alone.

In [8]:

peer_rows = []
for peer in cfg.US_PEERS:
    peer_ret = own_returns(peer)
    peer_est_dates = peer_ret.index[peer_ret.index <= window_end][-cfg.ESTIMATION_WINDOW_LENGTH:]
    peer_model = es.estimate_market_model(peer_ret, gspc_ret, peer_est_dates)
    peer_ar = es.abnormal_returns(peer_ret, gspc_ret, peer_model)
    wdates = es.event_window_dates(peer_ret.index, events["announcement (26-Apr-2026)"], -1, 1)
    res = es.car_with_tstat(peer_ar, wdates, peer_model["resid_std"])
    peer_rows.append({"peer": peer, "CAR_(-1,+1)": res["CAR"], "t_stat": res["t_stat"], "p_value": res["p_value"]})

peer_spillover = pd.DataFrame(peer_rows).set_index("peer")
peer_spillover.round(4)


,"CAR_(-1,+1)",t_stat,p_value
peer,,,
MRK,-0.0430,-1.3227,0.3169
PFE,-0.0081,-0.2980,0.7938
VTRS,0.0012,0.0325,0.9770
TEVA,0.0059,0.1319,0.9071


## Summary

In [9]:

ogn_car_ann_5 = ogn_car[(ogn_car["event"]=="announcement (26-Apr-2026)") & (ogn_car["window"]=="(-5,5)")].iloc[0]
sun_car_ann_5 = sun_car[(sun_car["event"]=="announcement (26-Apr-2026)") & (sun_car["window"]=="(-5,5)")].iloc[0]

summary = pd.Series({
    "ogn_car_announcement_pm5d": ogn_car_ann_5["CAR"],
    "ogn_car_announcement_tstat": ogn_car_ann_5["t_stat"],
    "sun_car_announcement_pm5d": sun_car_ann_5["CAR"],
    "sun_car_announcement_tstat": sun_car_ann_5["t_stat"],
    "leak_volume_ratio": vol_result["ratio"],
    "leak_volume_zscore": vol_result["z_score"],
    "pct_move_pre_announcement_simple": pct_pre_simple,
    "pct_move_pre_announcement_log": pct_pre_log,
    "peer_spillover_significant_count": int((peer_spillover["p_value"] < 0.05).sum()),
})
summary.to_csv(cfg.DATA_FINAL / "event_study_summary.csv", header=["value"])
summary.round(4)


ogn_car_announcement_pm5d           0.3303
ogn_car_announcement_tstat          2.5241
sun_car_announcement_pm5d           0.0962
sun_car_announcement_tstat          2.6361
leak_volume_ratio                   1.0431
leak_volume_zscore                  0.0522
pct_move_pre_announcement_simple    0.6960
pct_move_pre_announcement_log       0.7580
peer_spillover_significant_count    0.0000
dtype: float64